# Module 2 · Lesson 06: Output Parsing

LLMs return **strings**, but your app needs **structured data**.
This lesson covers reliable techniques for parsing LLM outputs.

## What you will learn
1. **JSON parsing** — the most common pattern
2. **Delimiter-based** extraction
3. **Regex** for flexible parsing
4. **Multi-section** parsing
5. **Error handling** and fallback strategies

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [2]:
def ask_open_ai(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"**The function has run**"))

**The function has run**

---
## 1. JSON Parsing — The Go-To Pattern

Ask the model to return JSON, then parse it in Python:

In [4]:
prompt = """Analyse this product review and return JSON:

Review: "The laptop is blazing fast and the screen is gorgeus, but the battery only last 4 hours and it runs hot during gaming.

Return JSON with: sentiments (positive/negative/mixed), pros (list), con (list), rating (1-5)

Return ONLY the JSON, no explanation."""

#raw = ask_open_ai(prompt)

def parse_json(text: str) -> dict:
    import re
    import json
    """Robustly parse JSON from LLM output."""
    text = text.strip()
    # Remove markdown code blocks
    if "```" in text:
        text = re.sub(r'```(?:json)?\n?', '', text).strip()
    return json.loads(text)
 
parsed = parse_json(raw)
 
display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))

```json
{
  "sentiments": "mixed",
  "pros": [
    "blazing fast",
    "gorgeous screen"
  ],
  "cons": [
    "battery only lasts 4 hours",
    "runs hot during gaming"
  ],
  "rating": 3
}
```

### Below quote an "ultra bulletproof" function for JSON LLM responses

In [ ]:
import ast
import json
import re
from typing import Any


def parse_json_ultra(text: str) -> Any:
    """
    Ultra-robust parser for JSON-ish LLM output.

    Handles:
    - Raw valid JSON
    - ```json fenced blocks
    - Extra text before/after JSON
    - JSON objects or arrays embedded in text
    - Common LLM mistakes:
        * single quotes instead of double quotes
        * trailing commas
        * Python literals: True / False / None
        * Python dict/list output

    Returns:
        Parsed Python object (dict, list, str, int, etc.)

    Raises:
        ValueError if no parseable JSON-like structure is found.
    """
    if not isinstance(text, str):
        raise TypeError("parse_json_ultra expects a string")

    text = text.strip()

    # 1) Fast path: exact valid JSON
    result = _try_json_loads(text)
    if result is not None:
        return result

    # 2) Extract fenced code blocks and try each one
    fenced_blocks = re.findall(
        r"```(?:json|javascript|js|python)?\s*(.*?)\s*```",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    for block in fenced_blocks:
        candidate = block.strip()
        result = _parse_jsonish_candidate(candidate)
        if result is not None:
            return result

    # 3) Try parsing the whole text as "JSON-ish"
    result = _parse_jsonish_candidate(text)
    if result is not None:
        return result

    # 4) Extract first balanced object/array from surrounding text
    candidate = _extract_first_balanced_json_like(text)
    if candidate:
        result = _parse_jsonish_candidate(candidate)
        if result is not None:
            return result

    # 5) Try all balanced candidates (in case first one is wrong)
    for candidate in _extract_all_balanced_json_like(text):
        result = _parse_jsonish_candidate(candidate)
        if result is not None:
            return result

    raise ValueError("Could not parse JSON or JSON-like content from text")


def _try_json_loads(s: str) -> Any | None:
    try:
        return json.loads(s)
    except Exception:
        return None


def _parse_jsonish_candidate(candidate: str) -> Any | None:
    """
    Try multiple increasingly permissive strategies on a candidate string.
    """
    candidate = candidate.strip()
    if not candidate:
        return None

    # A) Strict JSON first
    result = _try_json_loads(candidate)
    if result is not None:
        return result

    # B) Clean common JSON issues, then try JSON again
    cleaned = _cleanup_jsonish(candidate)
    result = _try_json_loads(cleaned)
    if result is not None:
        return result

    # C) Safe Python-literal fallback (great for {'a': 1}, True, None, etc.)
    result = _try_literal_eval(candidate)
    if result is not None:
        return result

    result = _try_literal_eval(cleaned)
    if result is not None:
        return result

    return None


def _cleanup_jsonish(s: str) -> str:
    """
    Clean common LLM mistakes while trying not to break valid content.
    """
    s = s.strip()

    # Remove leading/trailing markdown fences if somehow still present
    s = re.sub(r"^```(?:json|javascript|js|python)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    # Remove trailing commas before } or ]
    s = re.sub(r",\s*([}\]])", r"\1", s)

    # Convert Python literals to JSON literals outside strings
    s = _replace_outside_strings(s, r"\bTrue\b", "true")
    s = _replace_outside_strings(s, r"\bFalse\b", "false")
    s = _replace_outside_strings(s, r"\bNone\b", "null")

    return s


def _try_literal_eval(s: str) -> Any | None:
    """
    Safe fallback for Python-literal-like outputs:
    - {'a': 1}
    - [1, 2, 3]
    - True / False / None
    """
    try:
        return ast.literal_eval(s)
    except Exception:
        return None


def _extract_first_balanced_json_like(text: str) -> str | None:
    """
    Extract the first balanced {...} or [...] block from text,
    respecting quoted strings and escapes.
    """
    candidates = _extract_all_balanced_json_like(text)
    return candidates[0] if candidates else None


def _extract_all_balanced_json_like(text: str) -> list[str]:
    """
    Extract all top-level balanced {...} and [...] blocks from text.
    Useful when the first candidate is not actually the payload.
    """
    results = []
    n = len(text)

    for start in range(n):
        if text[start] not in "{[":
            continue

        opening = text[start]
        closing = "}" if opening == "{" else "]"

        depth = 0
        in_string = False
        escape = False

        for i in range(start, n):
            ch = text[i]

            if in_string:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_string = False
                elif ch == "'" and _is_probably_single_quoted_context(text, start):
                    # Best-effort support for single-quoted Python-ish strings
                    in_string = False
            else:
                if ch == '"':
                    in_string = True
                elif ch == "'":
                    # Best-effort support for Python-like dict strings
                    in_string = True
                elif ch == opening:
                    depth += 1
                elif ch == closing:
                    depth -= 1
                    if depth == 0:
                        results.append(text[start:i + 1].strip())
                        break

    # De-duplicate while preserving order
    seen = set()
    unique = []
    for item in results:
        if item not in seen:
            seen.add(item)
            unique.append(item)

    return unique


def _replace_outside_strings(s: str, pattern: str, replacement: str) -> str:
    """
    Replace regex pattern only when outside quoted strings.
    Supports both single and double quotes.
    """
    out = []
    i = 0
    n = len(s)
    in_string = False
    quote_char = None
    escape = False

    while i < n:
        ch = s[i]

        if in_string:
            out.append(ch)
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == quote_char:
                in_string = False
                quote_char = None
            i += 1
        else:
            if ch in ("'", '"'):
                in_string = True
                quote_char = ch
                out.append(ch)
                i += 1
            else:
                # Find next quote boundary so we can safely regex this chunk
                j = i
                while j < n and s[j] not in ("'", '"'):
                    j += 1
                chunk = s[i:j]
                chunk = re.sub(pattern, replacement, chunk)
                out.append(chunk)
                i = j

    return "".join(out)


def _is_probably_single_quoted_context(text: str, start: int) -> bool:
    """
    Best-effort helper: if the candidate starts with { or [ and contains single quotes,
    it's likely Python-literal-like content. This function intentionally stays simple.
    """
    snippet = text[start:start + 200]
    return "'" in snippet

---
## 2. Delimiter-Based Extraction

Use **unique delimiters** to separate sections reliably:

In [ ]:
prompt = """Analyse this code and provide feedback in this exact format:

---SUMMARY---

One-line summary of the code

---ISSUES---

List each issue on a new line

---IMPROVED---

The improved code

---END---

Code:
```python
def avg(nums):
    return sum(nums)/len(nums)
```
"""

result = ask_open_ai(prompt)
display(Markdown(result))

---SUMMARY---
Calculates the average of a list of numbers.
---ISSUES---
1. The function does not handle the case where the input list is empty, which would raise a `ZeroDivisionError`.
2. There is no type checking for the input, which could lead to errors if the input is not a list of numbers.
---IMPROVED---
```python
def avg(nums):
    if not nums:
        raise ValueError("The input list is empty.")
    if not all(isinstance(num, (int, float)) for num in nums):
        raise TypeError("All elements in the input list must be numbers.")
    return sum(nums) / len(nums)
```
---END---

---
## 3. Regex Extraction

When the output format is less predictable, **regex** provides flexibility:

In [6]:
prompt = """List 3 Python packages with their version numbers and an one-line description.

Format each as : package_name (vX.Y.Z) - description.
"""

result = ask_open_ai(prompt)

print(f"Raw output:\n{result}\n")

Raw output:
Here are three Python packages with their version numbers and descriptions:

1. NumPy (v1.23.5) - A fundamental package for numerical computing in Python, providing support for arrays and matrices.
2. Pandas (v1.5.1) - A powerful data analysis and manipulation library for structured data, offering data frames and series.
3. Matplotlib (v3.6.0) - A plotting library for creating static, animated, and interactive visualizations in Python.



In [8]:
import re 

pattern = r'(\w+)\s*\(v?([\d.]+)\)\s*[-–]\s*(.+)'
matches = re.findall(pattern, result)
 
if matches:
    print(f"{'Package':<15} {'Version':<10} {'Description'}")
    print("─" * 60)
    for name, version, desc in matches:
        print(f"{name:<15} {version:<10} {desc.strip()}")
else:
    print("⚠️ Regex didn't match — format may have varied")

Package         Version    Description
────────────────────────────────────────────────────────────
NumPy           1.23.5     A fundamental package for numerical computing in Python, providing support for arrays and matrices.
Pandas          1.5.1      A powerful data analysis and manipulation library for structured data, offering data frames and series.
Matplotlib      3.6.0      A plotting library for creating static, animated, and interactive visualizations in Python.


---
## 4. OpenAI Structured Output (response_format)

OpenAI offers `response_format={"type": "json_object"}` to **guarantee** valid JSON:

In [9]:
response = client.chat.completions.create(
    model= OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "You analyse text and return JSON."},
        {"role": "user", "content": "Analyse: 'This restaurant has amazing pasta but terrirble service.' Return JSON with: sentiment, food_quality (1-5), one-line summary"}
        ],
        response_format= {"type": "json_object"},
        max_tokens= 250 
)

parsed = json.loads(response.choices[0].message.content)

### We appear the results in 3 different ways

In [10]:
parsed

{'sentiment': 'mixed',
 'food_quality': 4,
 'one_line_summary': 'The restaurant offers great pasta but suffers from poor service.'}

In [11]:
print(parsed)

{'sentiment': 'mixed', 'food_quality': 4, 'one_line_summary': 'The restaurant offers great pasta but suffers from poor service.'}


In [12]:
display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))

```json
{
  "sentiment": "mixed",
  "food_quality": 4,
  "one_line_summary": "The restaurant offers great pasta but suffers from poor service."
}
```

> 💡 **Best Practice:** Use `response_format={"type": "json_object"}` in production.
> It guarantees valid JSON output — no manual parsing errors!

---
## 5. Error Handling & Fallbacks

---
## 6. Advanced: Structured JSON Game State

Real-world AI apps often need LLMs to return **complex, multi-field JSON** that drives application logic.

This pattern — inspired by community projects like AI dungeon games — uses the **system prompt as a contract** 
that defines the exact JSON schema the model must follow.

> 💡 **Why this matters:** This is the same pattern behind OpenAI's function-calling / tool-use feature (Module 5).
> Mastering JSON contracts now will make agents much easier to build later.

In [13]:
quiz_system = """You are QuizMaster, and AI that generates quiz questions.

You MUST always respond with JSON object with this EXACT structure:
{
    "question": "The quiz question text",
    "options": ["A) ...", "B) ...", "C) ...", "D) ..."],
    "correct_index": 0,
    "difficulty": "easy | medium | hard",
    "explanation": "Why the correct answer is right (1-2 sentences)",
    "topic": "The topic category"
}

Rules:
- Always return exactly 4 options.
- correct_index is 0-based (0=A, 1=B, 2=C, 3=D).  
- Return ONLY the JSON, no extra text.
"""

In [ ]:
raw_quiz = ask_open_ai("Generate a medium-difficulty question about Python programming",
                       system_content= quiz_system,
                       max_resp_tokens= 500)

print(raw_quiz)

{
    "question": "What is the output of the following code: print(type([]) is list)?",
    "options": ["A) True", "B) False", "C) None", "D) Error"],
    "correct_index": 0,
    "difficulty": "medium",
    "explanation": "The output is True because the type of an empty list is indeed 'list', and the 'is' operator checks for object identity.",
    "topic": "Python Programming"
}


### We need to create validations for the LLM responses, do not let in the lucky the responses.

In [16]:
# ── Robust parser with fallbacks ─────────────────────
def robust_parse(text: str) -> dict:
    """Try multiple parsing strategies."""
    # Strategy 1: Direct JSON
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
 
    # Strategy 2: Extract from code blocks
    match = re.search(r'```(?:json)?\s*(.+?)\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
 
    # Strategy 3: Find JSON-like content
    match = re.search(r'\{.+\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
 
    # Strategy 4: Return raw text as fallback
    return {"raw": text, "parse_error": True}
 
# Test with messy outputs
test_cases = [
    '{"name": "test", "value": 42}',
    'Here is the result:\n```json\n{"name": "test"}\n```',
    'The answer is {"result": true} as expected.',
    'No JSON here, just text.',
]
 
for text in test_cases:
    parsed = robust_parse(text)
    status = "Fallback" if parsed.get("parse_error") else "Parsed"
    print(f"{status}: {text[:40]+'...':<45} → {parsed}")

Parsed: {"name": "test", "value": 42}...              → {'name': 'test', 'value': 42}
Parsed: Here is the result:
```json
{"name": "te...   → {'name': 'test'}
Parsed: The answer is {"result": true} as expect...   → {'result': True}
Fallback: No JSON here, just text....                   → {'raw': 'No JSON here, just text.', 'parse_error': True}


In [17]:
# ── Step 3: Parse and validate the structured output ────
 
def validate_quiz(data: dict) -> dict:
    """Validate quiz JSON has all required fields with correct types."""
    errors = []
 
    required = {"question": str, "options": list, "correct_index": int,
                "difficulty": str, "explanation": str, "topic": str}
 
    for field, expected_type in required.items():
        if field not in data:
            errors.append(f"Missing field: '{field}'")
        elif not isinstance(data[field], expected_type):
            errors.append(f"Wrong type for '{field}': expected {expected_type.__name__}")
 
    if "options" in data and isinstance(data["options"], list):
        if len(data["options"]) != 4:
            errors.append(f"Expected 4 options, got {len(data['options'])}")
 
    if "correct_index" in data and isinstance(data["correct_index"], int):
        if data["correct_index"] not in range(4):
            errors.append(f"correct_index {data['correct_index']} out of range 0-3")
 
    if "difficulty" in data and data["difficulty"] not in ("easy", "medium", "hard"):
        errors.append(f"Invalid difficulty: '{data['difficulty']}'")
 
    return {"valid": len(errors) == 0, "errors": errors}
 
# Parse and validate
quiz = robust_parse(raw_quiz)
validation = validate_quiz(quiz)
 
if validation["valid"]:
    print(" Quiz question is valid!\n")
    display(Markdown(f"###  {quiz['question']}\n\n" +
            "\n".join(f"  {opt}" for opt in quiz['options']) +
            f"\n\n**Difficulty:** {quiz['difficulty']} · **Topic:** {quiz['topic']}\n\n" +
            f"**Answer:** {quiz['options'][quiz['correct_index']]}\n\n" +
            f"*{quiz['explanation']}*"))
else:
    print(" Validation failed:")
    for err in validation["errors"]:
        print(f"  • {err}")

 Quiz question is valid!



###  What is the output of the following code: print(type([]) is list)?

  A) True
  B) False
  C) None
  D) Error

**Difficulty:** medium · **Topic:** Python Programming

**Answer:** A) True

*The output is True because the type of an empty list is indeed 'list', and the 'is' operator checks for object identity.*

In [20]:
# ── Step 4: Build a mini quiz round (3 questions) ────────
 
topics = ["Python data types", "JavaScript closures", "SQL joins"]
 
print("=" * 50)
print("🎮 QUICK QUIZ — 3 Questions")
print("=" * 50)
 
for i, topic in enumerate(topics, 1):
    raw = ask_open_ai(f"Generate an easy question about {topic}.",
             system_content=quiz_system, max_resp_tokens=300)
    q = robust_parse(raw)
    v = validate_quiz(q)
 
    if not v["valid"]:
        print(f"\n⚠️ Q{i}: Skipped (invalid format: {v['errors']})")
        continue
 
    print(f"\n📝 Q{i}: {q['question']}")
    for opt in q["options"]:
        print(f"    {opt}")
    print(f"   ✅ Answer: {q['options'][q['correct_index']]}")
    print(f"   💡 {q['explanation']}")
 
print(f"\n{'=' * 50}")
print(f"Generated {len(topics)} structured quiz questions with validation!")

🎮 QUICK QUIZ — 3 Questions

📝 Q1: What data type is used to represent a sequence of characters in Python?
    A) String
    B) Integer
    C) List
    D) Dictionary
   ✅ Answer: A) String
   💡 A string is specifically designed to hold a sequence of characters in Python, making it the correct answer.

📝 Q2: What is a closure in JavaScript?
    A) A function that remembers its lexical scope even when the function is executed outside that scope.
    B) A type of loop that executes a block of code multiple times.
    C) An object that holds key-value pairs.
    D) A method for defining new array elements.
   ✅ Answer: A) A function that remembers its lexical scope even when the function is executed outside that scope.
   💡 A closure is a function that retains access to its lexical scope, even when called outside that scope. This allows the function to remember the environment in which it was created.

📝 Q3: What type of SQL join returns all records from the left table and matched records f

> 🔑 **Key insight:** The system prompt acts as a *schema contract*. The LLM "agrees" to the format,
> and your code validates compliance. This is exactly how **function-calling** and **tool-use** work 
> in Module 5 — the schema tells the model what shape to return, and your code uses the result.

> **Exercise:** Try modifying the quiz system prompt to add a `"hint"` field. 
> Update `validate_quiz()` to check for it. Does the model comply?

---
## 6. Advanced: Structured JSON Game State

Real-world AI apps often need LLMs to return **complex, multi-field JSON** that drives application logic.

This pattern — inspired by community projects like AI dungeon games — uses the **system prompt as a contract** 
that defines the exact JSON schema the model must follow.

> 💡 **Why this matters:** This is the same pattern behind OpenAI's function-calling / tool-use feature (Module 5).
> Mastering JSON contracts now will make agents much easier to build later.

> 🔑 **Key insight:** The system prompt acts as a *schema contract*. The LLM "agrees" to the format,
> and your code validates compliance. This is exactly how **function-calling** and **tool-use** work 
> in Module 5 — the schema tells the model what shape to return, and your code uses the result.

> **Exercise:** Try modifying the quiz system prompt to add a `"hint"` field. 
> Update `validate_quiz()` to check for it. Does the model comply?

---
## Key Takeaways 📝

| Method | Reliability | Best For |
|--------|-------------|----------|
| `response_format` | ⭐⭐⭐⭐⭐ | Production JSON |
| JSON prompt | ⭐⭐⭐⭐ | Most structured output |
| Delimiters | ⭐⭐⭐ | Multi-section text |
| Regex | ⭐⭐ | Flexible patterns |
| Robust parser | Always | Fallback strategy |

### Production Checklist
1. Use `response_format={"type": "json_object"}` when available
2. Always wrap parsing in try/except
3. Implement fallback strategies
4. Log parse failures for debugging

---
**Next:** `07_resume_customizer.ipynb` 